In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img

# Find local data when running from the project root or the 5 folder
root = Path("LP IV Updated Datasets/Object Detection")
if not root.is_dir():
    root = Path("../LP IV Updated Datasets/Object Detection")

dataset_path = root / "fashion mnist"
weights_path = root / "vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"

# Select only the ten Fashion MNIST categories
classes = [
    "Ankle-boot", "Bag", "Coat", "Dress", "Pullover",
    "Sandal", "Shirt", "Sneaker", "T-shirt-top", "Trouser"
]

# Load and preprocess images
# The validation images are held out from training
data = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train = data.flow_from_directory(
    str(dataset_path),
    classes=classes,
    target_size=(224, 224),
    batch_size=16,
    subset="training",
    class_mode="sparse"
)

test = data.flow_from_directory(
    str(dataset_path),
    classes=classes,
    target_size=(224, 224),
    batch_size=16,
    subset="validation",
    class_mode="sparse",
    shuffle=False
)

# Load pretrained VGG16 and freeze its layers
base = VGG16(
    weights=str(weights_path),
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)
base.trainable = False

# Train a new classifier for this dataset
model = Sequential([
    base,
    Dense(train.num_classes, activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.fit(train, epochs=3)

# Fine-tune the last part of VGG16 with a small learning rate
base.trainable = True
for layer in base.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.fit(train, epochs=1)

# Evaluate the final model on held-out validation images
loss, accuracy = model.evaluate(test)
print("Loss:", loss)
print("Validation accuracy:", accuracy * 100, "%")


In [ ]:
# Display six random validation images with their true and predicted labels
import random
from tensorflow.keras.utils import img_to_array

class_names = list(test.class_indices)
indices = random.sample(range(test.samples), 6)

plt.figure(figsize=(12, 8))

for i in range(6):
    index = indices[i]
    image = load_img(test.filepaths[index], target_size=(224, 224))
    image_array = img_to_array(image).reshape(1, 224, 224, 3)
    image_array = preprocess_input(image_array)

    prediction = model.predict(image_array, verbose=0).argmax()
    actual = test.classes[index]

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(
        "True: " + class_names[actual] +
        "\nPredicted: " + class_names[prediction]
    )
    plt.axis("off")

plt.tight_layout()
plt.show()
